# 08 Hybrid Search Rrf Fusion

**Project:** Enterprise Multi-Document RAG Assistant

**Notebook:** `08-hybrid-search-rrf-fusion.ipynb`

In [74]:
import pickle
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

In [75]:
children_df = pd.read_csv("artifacts/child_chunks.csv")

parents_df = pd.read_csv("artifacts/parent_chunks.csv")

metadata_df = pd.read_csv("artifacts/document_metadata.csv")

children_df = children_df.merge(
    metadata_df[["document_id", "department"]], on="document_id", how="left"
)

print(children_df.shape)
print(parents_df.shape)

(16, 5)
(6, 3)


In [76]:
parent_lookup = dict(zip(parents_df["parent_id"], parents_df["content"]))


def get_parent_context(parent_id):

    return parent_lookup.get(parent_id, "")

In [77]:
router_keywords = {
    "hr": ["employee", "insurance", "benefits", "healthcare", "salary", "leave"],
    "finance": ["revenue", "profit", "financial", "margin", "expense", "budget"],
    "engineering": ["ai", "vector", "search", "retrieval", "rag", "infrastructure"],
}

In [78]:
def route_query(query):

    query = query.lower()

    scores = {}

    for department, keywords in router_keywords.items():

        score = 0

        for keyword in keywords:

            if keyword in query:

                score += 1

        scores[department] = score

    best_department = max(scores, key=scores.get)

    return best_department, scores

In [79]:
bm25_indexes = {}

department_chunks = {}

In [80]:
for department in children_df["department"].unique():

    subset = children_df[children_df["department"] == department].reset_index(drop=True)

    corpus = [text.lower().split() for text in subset["content"]]

    bm25_indexes[department] = BM25Okapi(corpus)

    department_chunks[department] = subset

print(bm25_indexes.keys())

dict_keys(['engineering', 'finance', 'hr'])


In [81]:
def bm25_department_search(query, department, top_k=10):

    bm25 = bm25_indexes[department]

    chunk_df = department_chunks[department]

    scores = bm25.get_scores(query.lower().split())

    chunk_df = chunk_df.copy()

    chunk_df["bm25_score"] = scores

    results = chunk_df.sort_values("bm25_score", ascending=False)

    return results.head(top_k)

In [82]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [83]:
vector_indexes = {}

chunk_mappings = {}

In [84]:
for department in children_df["department"].unique():

    subset = children_df[children_df["department"] == department].reset_index(drop=True)

    texts = subset["content"].tolist()

    embeddings = embedding_model.encode(texts, convert_to_numpy=True)

    embeddings = embeddings.astype("float32")

    faiss.normalize_L2(embeddings)

    index = faiss.IndexFlatIP(embeddings.shape[1])

    index.add(embeddings)

    vector_indexes[department] = index

    chunk_mappings[department] = subset

print(vector_indexes.keys())

dict_keys(['engineering', 'finance', 'hr'])


In [85]:
def semantic_search(query, department, top_k=10):

    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    query_embedding = query_embedding.astype("float32")

    faiss.normalize_L2(query_embedding)

    index = vector_indexes[department]

    scores, ids = index.search(query_embedding, top_k)

    mapping = chunk_mappings[department]

    results = []

    for score, idx in zip(scores[0], ids[0]):

        row = mapping.iloc[idx]

        results.append(
            {
                "parent_id": row["parent_id"],
                "content": row["content"],
                "score": float(score),
            }
        )

    return results

In [86]:
def semantic_search(query, department, top_k=10):

    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    query_embedding = query_embedding.astype("float32")

    faiss.normalize_L2(query_embedding)

    index = vector_indexes[department]

    scores, ids = index.search(query_embedding, top_k)

    mapping = chunk_mappings[department]

    results = []

    for score, idx in zip(scores[0], ids[0]):

        row = mapping.iloc[idx]

        results.append(
            {
                "parent_id": row["parent_id"],
                "content": row["content"],
                "score": float(score),
            }
        )

    return results

In [87]:
def bm25_ranked_search(query, department, top_k=10):

    results = bm25_department_search(query, department, top_k)

    ranked = []

    for rank, (_, row) in enumerate(results.iterrows(), start=1):

        ranked.append({"parent_id": row["parent_id"], "rank": rank})

    return ranked

In [88]:
def semantic_ranked_search(query, department, top_k=10):

    results = semantic_search(query, department, top_k)

    ranked = []

    for rank, row in enumerate(results, start=1):

        ranked.append({"parent_id": row["parent_id"], "rank": rank})

    return ranked

In [89]:
def reciprocal_rank_fusion(bm25_results, semantic_results, k=60):

    scores = {}

    for result in bm25_results:

        doc_id = result["parent_id"]

        rank = result["rank"]

        score = 1 / (k + rank)

        scores[doc_id] = scores.get(doc_id, 0) + score

    for result in semantic_results:

        doc_id = result["parent_id"]

        rank = result["rank"]

        score = 1 / (k + rank)

        scores[doc_id] = scores.get(doc_id, 0) + score

    return scores

In [90]:
def sort_rrf_results(scores):

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [91]:
def hybrid_search(query, top_k=5):

    department, router_scores = route_query(query)

    bm25_results = bm25_ranked_search(query, department, top_k=20)

    semantic_results = semantic_ranked_search(query, department, top_k=20)

    fused_scores = reciprocal_rank_fusion(bm25_results, semantic_results)

    ranked_docs = sort_rrf_results(fused_scores)

    final_results = []

    for parent_id, score in ranked_docs[:top_k]:

        final_results.append(
            {
                "parent_id": parent_id,
                "rrf_score": round(score, 4),
                "context": get_parent_context(parent_id),
            }
        )

    return {
        "department": department,
        "router_scores": router_scores,
        "results": final_results,
    }

In [92]:
results = hybrid_search("What AI infrastructure projects are planned?")

print("Department:", results["department"])

for item in results["results"]:

    print("\n")
    print("=" * 60)

    print("RRF Score:", item["rrf_score"])

    print(item["context"][:300])

Department: engineering


RRF Score: 0.2692
er deployment planned.
Enterprise RAG platform launch Q3.
Agentic workflows under evaluation.
Component | Status
Vector Search | Production
Hybrid Retrieval | Testing
Cross Encoder | Planned



RRF Score: 0.0958
AI Platform Roadmap
Search Infrastructure
Retrieval Systems
Deployment Roadmap
Vector search infrastructure rollout.
Search latency improved significantly.
New indexing pipeline deployed.
Hybrid retrieval implementation.
Semantic search quality improved.
Knowledge base coverage expanded.
Cross encod
